In [10]:
import pyodbc

# اتصال به SQL Server
conn = pyodbc.connect(
    # 'DRIVER={SQL Server};'
    # 'SERVER=MKZ-DSAS\\DSAS;'
    # 'DATABASE=DSAS;'
    # 'UID=datadriven;'
    # 'PWD=5Rdx@4Rfv1355'
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'UID=rezapishva;'
    'PWD=5rdx@4rfv1355'
)

cursor = conn.cursor()

# لیست AssetIDهایی که می‌خوای بررسی بشن
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]

# لیست برای ذخیره مقادیر Value
values = []

# اجرای کوئری برای هر AssetID و دریافت فقط Value
for asset_id in asset_ids:
    query = f"""
        SELECT TOP 1 [Value]
        FROM [PDA].[Periodic_Values]
        WHERE UnitID = 11 AND AssetID = {asset_id}
        ORDER BY DateTime DESC
    """
    cursor.execute(query)
    row = cursor.fetchone()
    if row:
        values.append(row.Value)
    else:
        values.append(None)  


value_8341, value_8342, value_8343, value_8344, value_8346, value_9286, value_9287 = values

# نمایش مقادیر
print("✅ مقادیر آخرین رکوردها برای UnitID=11:")
print(f"AssetID 8341 → Value: {value_8341}")
print(f"AssetID 8342 → Value: {value_8342}")
print(f"AssetID 8343 → Value: {value_8343}")
print(f"AssetID 8344 → Value: {value_8344}")
print(f"AssetID 8346 → Value: {value_8346}")
print(f"AssetID 9286 → Value: {value_9286}")
print(f"AssetID 9287 → Value: {value_9287}")

# بستن اتصال
cursor.close()
conn.close()




import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import joblib

# بارگذاری اجزای مدل
scaler = joblib.load('scaler.pkl')
dbscan = joblib.load('dbscan_model.pkl')
cluster_points = np.load('cluster_points.npy')

# تابع تشخیص ناهنجاری با آستانه وزن
def is_anomalous(input_dict, threshold=10.0):
    input_df = pd.DataFrame([input_dict])
    scaled_input = scaler.transform(input_df)
    label = dbscan.fit_predict(scaled_input)[0]

    if label != -1:
        return {'is_anomaly': False, 'anomaly_weight': 0.0}

    # محاسبه فاصله از نزدیک‌ترین خوشه
    distance = pairwise_distances(scaled_input, cluster_points).min()
    anomaly_weight = distance

    # بررسی آستانه
    is_anomaly = anomaly_weight > threshold
    return {'is_anomaly': is_anomaly, 'anomaly_weight': anomaly_weight}

# مثال استفاده
sample_input = {
    'AssetID_8341': value_8341,
    'AssetID_8342': value_8342,
    'AssetID_8343': value_8343,
    'AssetID_8344': value_8344,
    'AssetID_8346': value_8346,
    'AssetID_9286': value_9286,
    'AssetID_9287': value_9287
}

result = is_anomalous(sample_input)
print(result)



import mysql.connector
from datetime import datetime

# اتصال به دیتابیس MySQL
conn = mysql.connector.connect(
    host='127.0.0.1',
    port=3306,
    user='root',
    password='',  
    database='dsas'
)

cursor = conn.cursor()

# مقادیر ورودی
inputs = [value_8341,value_8342,value_8343,value_8344,value_8346,value_9286,value_9287]
anomaly_weight = result['anomaly_weight']  # مقدار score
results = "Normal" if anomaly_weight < 5 else "Abnormal"
model_name = "Anomaly detection for lube oil system"
unitID = 11
system = "dbscan clustering weighted by computing distance from clusters "
score = anomaly_weight
created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
updated_at = created_at

print("1")

query = """
    INSERT INTO results_dsas_mhi_lube_oil_11 
    (inputs, results, model_name, unitID, system, score, created_at, updated_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""
print("2")

cursor.execute(query, (
    str(inputs),  # تبدیل لیست به رشته برای ذخیره در فیلد text یا varchar
    results,
    model_name,
    unitID,
    system,
    score,
    created_at,
    updated_at
))
print("3")
# ذخیره تغییرات
conn.commit()
print("4")
print("✅ داده با موفقیت ثبت شد.")

# بستن اتصال
cursor.close()
conn.close()


KeyboardInterrupt: 

In [1]:
import pandas as pd

# مسیر فایل اکسل
file_path = 'test.xlsx'  # ← اینجا نام فایل خودت رو بذار

# خواندن فایل اکسل
df = pd.read_excel(file_path)

# فرض بر اینه که ستون A بدون نام خاصی هست، پس از طریق اندیس صفر بهش دسترسی داریم
column_values = df.iloc[:, 0]  # ستون اول

# ساخت رشته نهایی با شماره ردیف و علامت "-"
combined_text = ''
for i, value in enumerate(column_values, start=1):
    combined_text += f'{i}-{value} '

# حذف فاصله‌ی اضافی آخر
combined_text = combined_text.strip()

# نمایش خروجی
print(combined_text)


1-درصد تبدیل نقاط قابل بهبود شناسایی شده به
 پروژه‌های بهبود سازمان (درصد) 2-درصد پیشرفت پروژه‌های بهبود سازمانی تعریف شده (درصد) 3-درصد اقدامات اصلاحی
 اجرا شده در موعد مقرر (درصد) 4-درصد اقدامات اصلاحی
 فاقد اثربخشی
 (درصد) 5-درصد مستندات 
بازنگری شده (درصد) 6-مدت‌زمان بازنگری مستندات (روز) 7-مدت‌زمان تهیه مستندات (روز) 8-درصد تعیین
 تکلیف سوابق 9-درصد اثربخشی خدمات نامنطبق رفع شده  10-متوسط زمان رفع ریشه ای خدمات نامنطبق  11-درصد اجرای مصوبات بازنگری مدیریت (درصد) 12-درصد تغییرات
 اجرا شده  13-درصد تغییرات
 اثربخش  14-درصد کاهش ریسک فرایندی (درصد) 15-درصد ریسک‌های فرایندی به‌روز شده (درصد) 16-درصد دانش‌های 
ارزیابی شده (درصد) 17-درصد دانش‌های 
کاربردی شده (درصد) 18-مدت‌زمان ارزیابی درخواست‌های دانش ثبت شده (درصد) 19-درصد تحقق درآمد (درصد) 20-درصد تحقق فروش (درصد) 21-ضریب خروج برنامه‌ریزی نشده ناشی (در تعهد بهره‌بردار) 22-نرخ خروج داخلی 23- نرخ خروج اضطراری 24-نرخ خروج با هماهنگی 25-نرخ خروج افزایش اجباری 
زمان تعمیرات 26-آمادگی محقق شده به آمادگی قراردادی (درصد) 27-آمادگی محقق شده ب

In [2]:
# pip install pandas openpyxl

In [8]:
import pandas as pd

# بارگذاری داده‌ها از شیت 1، شروع از ردیف 4
file_path = "kpis.xlsx"
df = pd.read_excel(file_path, sheet_name="1", header=None, skiprows=3, usecols="A:B")

# حذف ردیف‌هایی که شماره یا عنوان ندارن
df = df.dropna(subset=[0, 1])

# ترکیب شماره و عنوان شاخص به صورت سطری
for index, row in df.iterrows():
    print(f"{int(row[0])} - {row[1]}")


1 - اندازه‌گیری بموقع شاخص‌های فرایندی (روز)
2 - درصد تبدیل نقاط قابل بهبود شناسایی شده به
 پروژه‌های بهبود سازمان (درصد)
3 - درصد پیشرفت پروژه‌های بهبود سازمانی تعریف شده (درصد)
4 - درصد اقدامات اصلاحی
 اجرا شده در موعد مقرر (درصد)
5 - درصد اقدامات اصلاحی
 فاقد اثربخشی
 (درصد)
6 - درصد مستندات 
بازنگری شده (درصد)
7 - مدت‌زمان بازنگری مستندات (روز)
8 - مدت‌زمان تهیه مستندات (روز)
9 - درصد تعیین
 تکلیف سوابق
10 - درصد اثربخشی خدمات نامنطبق رفع شده 
11 - متوسط زمان رفع ریشه ای خدمات نامنطبق 
12 - درصد اجرای مصوبات بازنگری مدیریت (درصد)
13 - درصد تغییرات
 اجرا شده 
14 - درصد تغییرات
 اثربخش 
15 - درصد کاهش ریسک فرایندی (درصد)
16 - درصد ریسک‌های فرایندی به‌روز شده (درصد)
17 - درصد دانش‌های 
ارزیابی شده (درصد)
18 - درصد دانش‌های 
کاربردی شده (درصد)
19 - مدت‌زمان ارزیابی درخواست‌های دانش ثبت شده (درصد)
20 - درصد تحقق درآمد (درصد)
21 - درصد تحقق فروش (درصد)
22 - ضریب خروج برنامه‌ریزی نشده ناشی (در تعهد بهره‌بردار)
23 - نرخ خروج داخلی
24 -  نرخ خروج اضطراری
25 - نرخ خروج با هماهنگی
26 - نرخ خر

In [9]:
import openpyxl

# بارگذاری فایل اکسل
wb = openpyxl.load_workbook("kpis.xlsx")

# استخراج داده‌های شیت 1: شماره شاخص در ستون A، فرمول در ستون C
sheet1 = wb["1"]
formula_map = {}
for row in sheet1.iter_rows(min_row=2, values_only=True):
    index = row[0]  # ستون A
    formula = row[2]  # ستون C
    if index is not None:
        formula_map[str(index).strip()] = formula

# پردازش شیت‌های 2 تا 8
for i in range(2, 9):
    sheet = wb[str(i)]
    
    # اضافه کردن عنوان ستون جدید در سلول D1
    sheet["D1"] = "فرمول شاخص"
    
    # پیمایش ردیف‌ها از B2 به بعد
    for row in range(2, sheet.max_row + 1):
        index_cell = sheet[f"B{row}"]
        formula_cell = sheet[f"D{row}"]
        
        index_value = str(index_cell.value).strip() if index_cell.value is not None else ""
        formula_value = formula_map.get(index_value, "")
        
        formula_cell.value = formula_value

# ذخیره فایل
wb.save("kpis_updated.xlsx")


In [ ]:
import pyodbc

# مشخصات اتصال
server = 'MKZ-DSAS\\DSAS'  # نام سرور و
database = 'DSAS'          # نام دیتابیس
username = 'datadriven'            # یوزر فرضی (جایگزین کن)
password = '5Rdx@4Rfv1355'  # پسورد فرضی (جایگزین کن)

# رشته اتصال
conn_str = (
    'DRIVER={SQL Server};'
    'SERVER=MKZ-DSAS\\DSAS;'
    'DATABASE=DSAS;'
    'UID=datadriven;'
    'PWD=5Rdx@4Rfv1355'
)


try:
    # تلاش برای اتصال
    conn = pyodbc.connect(conn_str)
    print("اتصال موفق بود!")
    conn.close()  # بستن اتصال بعد از تست
except pyodbc.Error as e:
    print(f"خطا در اتصال: {e}")

اتصال موفق بود!


In [4]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'PostgreSQL ANSI(x64)', 'PostgreSQL Unicode(x64)', 'Amazon Redshift (x64)', 'SQL Server Native Client 10.0', 'MySQL ODBC 5.3 ANSI Driver', 'MySQL ODBC 5.3 Unicode Driver', 'ODBC Driver 18 for SQL Server']


In [ ]:
import os
import pyodbc
import pandas as pd

# -----------------------------
# تعریف متغیر واحد برای UnitID
# -----------------------------
unitID = 12
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]

# -----------------------------
# اتصال به SQL Server
# -----------------------------
conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'UID=rezapishva;'
    'PWD=5rdx@4rfv1355'
)

cursor = conn.cursor()

# -----------------------------
# گرفتن داده‌ها برای 100 روز مختلف
# -----------------------------
data_frames = []

for asset_id in asset_ids:
    query = f"""
        SELECT TOP 100 [AssetID], [Value], [RecordDate], [RecordTime], [DateTime]
        FROM [PDA].[Periodic_Values]
        WHERE UnitID = {unitID} AND AssetID = {asset_id}
        ORDER BY DateTime DESC
    """
    cursor.execute(query)
    rows = cursor.fetchall()
    cols = [col[0] for col in cursor.description]
    df = pd.DataFrame.from_records(rows, columns=cols)
    df.rename(columns={"Value": f"AssetID_{asset_id}"}, inplace=True)
    data_frames.append(df[["DateTime", f"AssetID_{asset_id}"]])

cursor.close()
conn.close()

# -----------------------------
# ترکیب همه AssetIDها در یک تایم‌فریم
# -----------------------------
final_df = data_frames[0]
for df in data_frames[1:]:
    final_df = pd.merge(final_df, df, on="DateTime", how="inner")

# -----------------------------
# نمایش خروجی
# -----------------------------
print("✅ تایم‌فریم ساخته شد:")
print(final_df.head(20))


KeyboardInterrupt: 

In [ ]:
import pyodbc
import pymysql
from datetime import datetime

# مرحله 1: اتصال به SQL Server
print("اتصال به SQL Server...")
conn_sqlserver = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'UID=rezapishva;'
    'PWD=5rdx@4rfv1355'
)

# مرحله 2: خواندن آخرین رکورد با شرط UnitID=11 و AssetID=8341
query = """
SELECT TOP 1 [UnitID], [Value], [RecordTime], [RecordDate], [DateTime], [TimeStamp]
FROM [PEGAH].[PDA].[Periodic_Values]
WHERE [UnitID] = 11 AND [AssetID] = 8341
ORDER BY [DateTime] DESC
"""

cursor_sql = conn_sqlserver.cursor()
cursor_sql.execute(query)
row = cursor_sql.fetchone()

if row is None:
    print("هشدار: هیچ رکوردی با شرط UnitID=11 و AssetID=8341 پیدا نشد.")
    conn_sqlserver.close()
    exit()

# استخراج داده‌ها
unit_id = row.UnitID
value_8341 = row.Value
record_time = row.RecordTime
record_date = row.RecordDate
date_time = row.DateTime
time_stamp = row.TimeStamp

print(f"داده خوانده شد: UnitID={unit_id}, Value={value_8341}, DateTime={date_time}")

# بستن اتصال SQL Server
cursor_sql.close()
conn_sqlserver.close()

# مرحله 3: اتصال به MySQL
print("اتصال به MySQL...")
conn_mysql = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="",
    database="dsas",
    charset="utf8mb4",
    cursorclass=pymysql.cursors.DictCursor  # برای راحتی کار با دیکشنری
)

# مرحله 4: درج داده در جدول MySQL
insert_query = """
INSERT INTO results_dsas_mhi_lube_oil_11 
(unitID, AssetID_8341, RecordTime, RecordDate, DateTime, TimeStamps)
VALUES (%s, %s, %s, %s, %s, %s)
"""

try:
    with conn_mysql.cursor() as cursor_mysql:
        cursor_mysql.execute(insert_query, (
            unit_id,
            value_8341,
            record_time,
            record_date,
            date_time,
            time_stamp
        ))
    conn_mysql.commit()
    print("رکورد با موفقیت در جدول results_dsas_mhi_lube_oil_11 درج شد.")
    print(f"داده درج شده: unitID={unit_id}, AssetID_8341={value_8341}, DateTime={date_time}")

except Exception as e:
    print(f"خطا در درج داده در MySQL: {e}")
    conn_mysql.rollback()

finally:
    conn_mysql.close()

print("عملیات تمام شد.")

اتصال به SQL Server...
داده خوانده شد: UnitID=11, Value=0.07, DateTime=2023-12-03 04:24:53
اتصال به MySQL...
رکورد با موفقیت در جدول results_dsas_mhi_lube_oil_11 درج شد.
داده درج شده: unitID=11, AssetID_8341=0.07, DateTime=2023-12-03 04:24:53
عملیات تمام شد.


In [2]:
import pyodbc
import pymysql
from datetime import datetime
import numpy as np

# ================== مرحله 1: اتصال به SQL Server ==================
print("اتصال به SQL Server...")
conn_sqlserver = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'UID=rezapishva;'
    'PWD=5rdx@4rfv1355'
)

# ================== مرحله 2: مقادیر آخرین رکوردها (همه 7 AssetID) ==================
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]
last_values = {}

print("خواندن آخرین مقادیر برای 7 AssetID از Periodic_Values...")
for asset_id in asset_ids:
    query = f"""
    SELECT TOP 1 [Value], [RecordTime], [RecordDate], [DateTime], [TimeStamp]
    FROM [PEGAH].[PDA].[Periodic_Values]
    WHERE [UnitID] = 11 AND [AssetID] = {asset_id}
    ORDER BY [DateTime] DESC
    """
    try:
        cursor = conn_sqlserver.cursor()
        cursor.execute(query)
        row = cursor.fetchone()
        if row:
            last_values[asset_id] = {
                'Value': row.Value,
                'RecordTime': row.RecordTime,
                'RecordDate': row.RecordDate,
                'DateTime': row.DateTime,
                'TimeStamp': row.TimeStamp
            }
        else:
            last_values[asset_id] = {'Value': None}
        cursor.close()
    except Exception as e:
        print(f"خطا در خواندن AssetID {asset_id}: {e}")
        last_values[asset_id] = {'Value': None}

# اگر هیچ داده‌ای نبود
if not any(v['Value'] is not None for v in last_values.values()):
    print("هشدار: هیچ داده‌ای از SQL Server خوانده نشد.")
    conn_sqlserver.close()
    exit()

# زمان آخرین رکورد (برای TimeStamps و DateTime)
latest_datetime = max((v['DateTime'] for v in last_values.values() if v['Value'] is not None), default=datetime.now())
latest_timestamp = last_values[8341].get('TimeStamp') or int(latest_datetime.timestamp())

# ================== مرحله 3: اتصال به MySQL و درج اولیه ==================
print("اتصال به MySQL...")
conn_mysql = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="",
    database="dsas",
    charset="utf8mb4",
    cursorclass=pymysql.cursors.DictCursor
)

insert_query = """
INSERT INTO results_dsas_mhi_lube_oil_11 (
    unitID,
    AssetID_8341, AssetID_8342, AssetID_8343, AssetID_8344, 
    AssetID_8346, AssetID_9286, AssetID_9287,
    RecordTime, RecordDate, DateTime, TimeStamps
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

values_to_insert = (
    11,
    last_values[8341]['Value'],
    None,  # AssetID_8342 → بعداً آپدیت میشه
    last_values[8343]['Value'],
    last_values[8344]['Value'],
    last_values[8346]['Value'],
    last_values[9286]['Value'],
    last_values[9287]['Value'],
    latest_datetime.strftime('%H:%M:%S'),
    latest_datetime.strftime('%Y-%m-%d'),
    latest_datetime,
    latest_timestamp
)

try:
    with conn_mysql.cursor() as cursor:
        cursor.execute(insert_query, values_to_insert)
    conn_mysql.commit()
    print("رکورد اولیه با موفقیت درج شد.")
except Exception as e:
    print(f"خطا در درج اولیه: {e}")
    conn_mysql.rollback()
    conn_sqlserver.close()
    conn_mysql.close()
    exit()

# ================== مرحله 4: آپدیت AssetID_8342 با شرط اختلاف TimeStamp < 1000 ==================
last_mysql_timestamp = latest_timestamp  # همون که درج کردیم

# کوئری برای پیدا کردن نزدیک‌ترین رکورد با اختلاف کمتر از 1000
query_near = """
SELECT TOP 1 [Value]
FROM [PEGAH].[PDA].[Periodic_Values]
WHERE [UnitID] = 11 
  AND [AssetID] = 8342
  AND ABS([TimeStamp] - %s) < 1000
ORDER BY ABS([TimeStamp] - %s)
"""

# کوئری برای میانگین 10 مقدار آخر
query_avg = """
SELECT AVG(CAST([Value] AS FLOAT)) as avg_value
FROM (
    SELECT TOP 10 [Value]
    FROM [PEGAH].[PDA].[Periodic_Values]
    WHERE [UnitID] = 11 AND [AssetID] = 8342
    ORDER BY [DateTime] DESC
) AS sub
"""

value_to_update = None

try:
    cursor_sql = conn_sqlserver.cursor()
    
    # اول: شرط اختلاف < 1000
    cursor_sql.execute(query_near, (last_mysql_timestamp, last_mysql_timestamp))
    row = cursor_sql.fetchone()
    
    if row and row.Value is not None:
        value_to_update = float(row.Value)
        print(f"مقدار نزدیک با اختلاف <1000 پیدا شد: {value_to_update}")
    else:
        # دوم: میانگین 10 مقدار آخر
        cursor_sql.execute(query_avg)
        avg_row = cursor_sql.fetchone()
        if avg_row and avg_row.avg_value is not None:
            value_to_update = round(float(avg_row.avg_value), 4)
            print(f"هیچ مقدار نزدیک پیدا نشد → میانگین 10 مقدار آخر: {value_to_update}")
        else:
            value_to_update = None
            print("هیچ داده‌ای برای AssetID=8342 پیدا نشد.")
    
    cursor_sql.close()

except Exception as e:
    print(f"خطا در خواندن داده برای AssetID_8342: {e}")
    value_to_update = None

# ================== مرحله 5: آپدیت فیلد AssetID_8342 در آخرین رکورد MySQL ==================
if value_to_update is not None:
    update_query = """
    UPDATE results_dsas_mhi_lube_oil_11
    SET AssetID_8342 = %s, updated_at = NOW()
    WHERE TimeStamps = %s AND unitID = 11
    """
    try:
        with conn_mysql.cursor() as cursor:
            cursor.execute(update_query, (value_to_update, last_mysql_timestamp))
        conn_mysql.commit()
        print(f"فیلد AssetID_8342 با مقدار {value_to_update} آپدیت شد.")
    except Exception as e:
        print(f"خطا در آپدیت MySQL: {e}")
        conn_mysql.rollback()
else:
    print("مقداری برای آپدیت AssetID_8342 پیدا نشد.")

# ================== بستن اتصالات ==================
conn_sqlserver.close()
conn_mysql.close()
print("عملیات با موفقیت به پایان رسید.")

اتصال به SQL Server...
خواندن آخرین مقادیر برای 7 AssetID از Periodic_Values...
اتصال به MySQL...
رکورد اولیه با موفقیت درج شد.
خطا در خواندن داده برای AssetID_8342: ('The SQL contains 0 parameter markers, but 2 parameters were supplied', 'HY000')
مقداری برای آپدیت AssetID_8342 پیدا نشد.
عملیات با موفقیت به پایان رسید.


In [7]:
import pyodbc
import pymysql
from datetime import datetime
import numpy as np

# ================== مرحله 1: اتصال به SQL Server ==================
print("اتصال به SQL Server...")
conn_sqlserver = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'UID=rezapishva;'
    'PWD=5rdx@4rfv1355'
)

# ================== مرحله 2: خواندن آخرین مقادیر برای همه 7 AssetID ==================
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]
last_values = {}

print("خواندن آخرین مقادیر برای 7 AssetID از Periodic_Values...")
for asset_id in asset_ids:
    query = f"""
    SELECT TOP 1 [Value], [RecordTime], [RecordDate], [DateTime], [TimeStamp]
    FROM [PEGAH].[PDA].[Periodic_Values]
    WHERE [UnitID] = 11 AND [AssetID] = {asset_id}
    ORDER BY [DateTime] DESC
    """
    try:
        cursor = conn_sqlserver.cursor()
        cursor.execute(query)
        row = cursor.fetchone()
        if row:
            last_values[asset_id] = {
                'Value': row.Value,
                'RecordTime': row.RecordTime,
                'RecordDate': row.RecordDate,
                'DateTime': row.DateTime,
                'TimeStamp': row.TimeStamp
            }
        else:
            last_values[asset_id] = {'Value': None}
        cursor.close()
    except Exception as e:
        print(f"خطا در خواندن AssetID {asset_id}: {e}")
        last_values[asset_id] = {'Value': None}

# زمان آخرین رکورد (برای TimeStamps)
latest_datetime = max(
    (v['DateTime'] for v in last_values.values() if v.get('DateTime')),
    default=datetime.now()
)
latest_timestamp = int(latest_datetime.timestamp())

# ================== مرحله 3: درج اولیه (همه فیلدها با مقدار یا NULL) ==================
print("درج رکورد اولیه در MySQL...")
conn_mysql = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="",
    database="dsas",
    charset="utf8mb4",
    cursorclass=pymysql.cursors.DictCursor
)

insert_query = """
INSERT INTO results_dsas_mhi_lube_oil_11 (
    unitID,
    AssetID_8341, AssetID_8342, AssetID_8343, AssetID_8344, 
    AssetID_8346, AssetID_9286, AssetID_9287,
    RecordTime, RecordDate, DateTime, TimeStamps
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

values_to_insert = (
    11,
    last_values[8341]['Value'],
    last_values[8342]['Value'],
    last_values[8343]['Value'],
    last_values[8344]['Value'],
    last_values[8346]['Value'],
    last_values[9286]['Value'],
    last_values[9287]['Value'],
    latest_datetime.strftime('%H:%M:%S'),
    latest_datetime.strftime('%Y-%m-%d'),
    latest_datetime,
    latest_timestamp
)

try:
    with conn_mysql.cursor() as cursor:
        cursor.execute(insert_query, values_to_insert)
    conn_mysql.commit()
    print("رکورد اولیه با موفقیت درج شد.")
except Exception as e:
    print(f"خطا در درج اولیه: {e}")
    conn_sqlserver.close()
    conn_mysql.close()
    exit()

# ================== مرحله 4: آپدیت فیلدهای NULL با منطق هوشمند (مثل 8342) ==================
print("آپدیت فیلدهای خالی با نزدیک‌ترین مقدار یا میانگین...")

# کوئری نزدیک‌ترین مقدار
query_near = """
SELECT TOP 1 [Value]
FROM [PEGAH].[PDA].[Periodic_Values]
WHERE [UnitID] = 11 AND [AssetID] = %s
  AND ABS([TimeStamp] - %s) < 1000
ORDER BY ABS([TimeStamp] - %s)
"""

# کوئری میانگین 10 مقدار آخر
query_avg = """
SELECT AVG(CAST([Value] AS FLOAT)) as avg_value
FROM (
    SELECT TOP 10 [Value]
    FROM [PEGAH].[PDA].[Periodic_Values]
    WHERE [UnitID] = 11 AND [AssetID] = %s
    ORDER BY [DateTime] DESC
) AS sub
"""

# مپینگ فیلدها
field_map = {
    8341: 'AssetID_8341',
    8342: 'AssetID_8342',
    8343: 'AssetID_8343',
    8344: 'AssetID_8344',
    8346: 'AssetID_8346',
    9286: 'AssetID_9286',
    9287: 'AssetID_9287'
}

cursor_sql = conn_sqlserver.cursor()

for asset_id, field_name in field_map.items():
    current_value = last_values[asset_id]['Value']
    
    # فقط اگر مقدار None یا خالی بود، آپدیت کن
    if current_value is not None:
        continue  # قبلاً مقدار داره، نیازی نیست

    value_to_update = None

    try:
        # 1. نزدیک‌ترین با اختلاف < 1000
        cursor_sql.execute(query_near, (asset_id, latest_timestamp, latest_timestamp))
        row = cursor_sql.fetchone()
        if row and row.Value is not None:
            value_to_update = float(row.Value)
            print(f"{field_name}: مقدار نزدیک پیدا شد → {value_to_update}")

        else:
            # 2. میانگین 10 مقدار آخر
            cursor_sql.execute(query_avg, (asset_id,))
            avg_row = cursor_sql.fetchone()
            if avg_row and avg_row.avg_value is not None:
                value_to_update = round(float(avg_row.avg_value), 4)
                print(f"{field_name}: میانگین 10 مقدار آخر → {value_to_update}")
            else:
                print(f"{field_name}: هیچ داده‌ای پیدا نشد.")
                continue

        # 3. آپدیت در MySQL
        update_query = f"""
        UPDATE results_dsas_mhi_lube_oil_11
        SET {field_name} = %s, updated_at = NOW()
        WHERE TimeStamps = %s AND unitID = 11
        """
        with conn_mysql.cursor() as cursor_mysql:
            cursor_mysql.execute(update_query, (value_to_update, latest_timestamp))
        conn_mysql.commit()
        print(f"{field_name} با موفقیت آپدیت شد.")

    except Exception as e:
        print(f"خطا در آپدیت {field_name}: {e}")

cursor_sql.close()
conn_sqlserver.close()
conn_mysql.close()

print("تمام عملیات با موفقیت انجام شد!")

اتصال به SQL Server...
خواندن آخرین مقادیر برای 7 AssetID از Periodic_Values...
درج رکورد اولیه در MySQL...
رکورد اولیه با موفقیت درج شد.
آپدیت فیلدهای خالی با نزدیک‌ترین مقدار یا میانگین...
تمام عملیات با موفقیت انجام شد!


In [1]:
import pyodbc
import pymysql
from datetime import datetime

# ================== اتصالات ==================
print("اتصال به SQL Server...")
conn_sql = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'UID=rezapishva;'
    'PWD=5rdx@4rfv1355'
)

print("اتصال به MySQL...")
conn_mysql = pymysql.connect(
    host="127.0.0.1", port=3306, user="root", password="", database="dsas",
    charset="utf8mb4", cursorclass=pymysql.cursors.DictCursor
)

# ================== لیست AssetIDها ==================
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]
field_map = {8341: 'AssetID_8341', 8342: 'AssetID_8342', 8343: 'AssetID_8343',
             8344: 'AssetID_8344', 8346: 'AssetID_8346', 9286: 'AssetID_9286', 9287: 'AssetID_9287'}

# ================== خواندن TimeStamps موجود در MySQL ==================
print("خواندن TimeStamps موجود در MySQL...")
existing_ts = set()
with conn_mysql.cursor() as c:
    c.execute("SELECT TimeStamps FROM results_dsas_mhi_lube_oil_11 WHERE unitID = 11")
    for row in c.fetchall():
        existing_ts.add(row['TimeStamps'])
print(f"{len(existing_ts)} رکورد موجود است.")

# ================== خواندن 10 رکورد آخر برای AssetID=8341 ==================
print("خواندن 10 رکورد آخر برای AssetID=8341...")
query_8341 = """
SELECT TOP 10 [Value], [RecordTime], [RecordDate], [DateTime], [TimeStamp]
FROM [PEGAH].[PDA].[Periodic_Values]
WHERE [UnitID] = 11 AND [AssetID] = 8341
ORDER BY [DateTime] DESC
"""
cursor = conn_sql.cursor()
cursor.execute(query_8341)
rows_8341 = cursor.fetchall()

if not rows_8341:
    print("هیچ داده‌ای برای AssetID=8341 پیدا نشد.")
    exit()

# ================== حلقه: درج هر رکورد + آپدیت 6 فیلد دیگر ==================
inserted_count = 0
for idx, row in enumerate(rows_8341):
    ts = row.TimeStamp
    if ts in existing_ts:
        print(f"رکورد {idx+1}: TimeStamps={ts} تکراری است → رد شد.")
        continue

    dt = row.DateTime
    print(f"\n--- درج رکورد {idx+1} | TimeStamps={ts} ---")

    # درج اولیه (فقط 8341 پر، بقیه NULL)
    insert_sql = """
    INSERT INTO results_dsas_mhi_lube_oil_11 
    (unitID, AssetID_8341, RecordTime, RecordDate, DateTime, TimeStamps)
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    try:
        with conn_mysql.cursor() as c:
            c.execute(insert_sql, (11, row.Value, 
                                 dt.strftime('%H:%M:%S'), dt.strftime('%Y-%m-%d'), dt, ts))
        conn_mysql.commit()
        print("رکورد درج شد.")
        inserted_count += 1
    except Exception as e:
        print(f"خطا در درج: {e}")
        continue

    # آپدیت 6 فیلد دیگر
    for aid in [8342, 8343, 8344, 8346, 9286, 9287]:
        field = field_map[aid]
        
        # 1. نزدیک‌ترین با اختلاف < 1000
        near_sql = f"""
        SELECT TOP 1 [Value] FROM [PEGAH].[PDA].[Periodic_Values]
        WHERE [UnitID]=11 AND [AssetID]=? AND ABS([TimeStamp] - ?) < 1000
        ORDER BY ABS([TimeStamp] - ?)
        """
        cursor.execute(near_sql, (aid, ts, ts))
        near_row = cursor.fetchone()

        if near_row and near_row.Value is not None:
            value = float(near_row.Value)
            source = "نزدیک"
        else:
            # 2. میانگین 10 آخر
            avg_sql = f"""
            SELECT AVG(CAST([Value] AS FLOAT)) FROM (
                SELECT TOP 10 [Value] FROM [PEGAH].[PDA].[Periodic_Values]
                WHERE [UnitID]=11 AND [AssetID]=?
                ORDER BY [DateTime] DESC
            ) sub
            """
            cursor.execute(avg_sql, (aid,))
            avg_row = cursor.fetchone()
            value = round(float(avg_row[0]), 4) if avg_row[0] else None
            source = "میانگین"

        if value is not None:
            update_sql = f"UPDATE results_dsas_mhi_lube_oil_11 SET {field} = %s WHERE TimeStamps = %s"
            try:
                with conn_mysql.cursor() as c:
                    c.execute(update_sql, (value, ts))
                conn_mysql.commit()
                print(f"   {field} = {value} ({source})")
            except Exception as e:
                print(f"   خطا در آپدیت {field}: {e}")
        else:
            print(f"   {field}: داده‌ای پیدا نشد.")

# ================== پایان ==================
cursor.close()
conn_sql.close()
conn_mysql.close()

print(f"\nنتیجه: {inserted_count} رکورد جدید با موفقیت درج و آپدیت شد.")

اتصال به SQL Server...
اتصال به MySQL...
خواندن TimeStamps موجود در MySQL...
10 رکورد موجود است.
خواندن 10 رکورد آخر برای AssetID=8341...
رکورد 1: TimeStamps=1701564893 تکراری است → رد شد.
رکورد 2: TimeStamps=1701550623 تکراری است → رد شد.
رکورد 3: TimeStamps=1701537147 تکراری است → رد شد.
رکورد 4: TimeStamps=1701521363 تکراری است → رد شد.
رکورد 5: TimeStamps=1701507631 تکراری است → رد شد.
رکورد 6: TimeStamps=1701493390 تکراری است → رد شد.
رکورد 7: TimeStamps=1701477893 تکراری است → رد شد.
رکورد 8: TimeStamps=1701464362 تکراری است → رد شد.
رکورد 9: TimeStamps=1701449934 تکراری است → رد شد.
رکورد 10: TimeStamps=1701435054 تکراری است → رد شد.

نتیجه: 0 رکورد جدید با موفقیت درج و آپدیت شد.


In [ ]:
import pyodbc
import pymysql
import time
from datetime import datetime
import threading
import signal
import sys

# ================== تنظیمات ==================
INTERVAL_SECONDS = 30 * 60  # هر 5 دقیقه
RUNNING = True  # برای توقف با Ctrl+C

# ================== اتصالات ==================
def get_sql_connection():
    return pyodbc.connect(
        'DRIVER={SQL Server};'
        'SERVER=10WKS-PISHVA;'
        'DATABASE=PEGAH;'
        'UID=rezapishva;'
        'PWD=5rdx@4rfv1355'
    )

def get_mysql_connection():
    return pymysql.connect(
        host="127.0.0.1", port=3306, user="root", password="", database="dsas",
        charset="utf8mb4", cursorclass=pymysql.cursors.DictCursor
    )

# ================== لیست AssetIDها ==================
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]
field_map = {8341: 'AssetID_8341', 8342: 'AssetID_8342', 8343: 'AssetID_8343',
             8344: 'AssetID_8344', 8346: 'AssetID_8346', 9286: 'AssetID_9286', 9287: 'AssetID_9287'}

# ================== تابع اصلی (همون کد قبلی) ==================
def run_task():
    print(f"\n{'='*60}")
    print(f"اجرای وظیفه: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}")

    conn_sql = None
    conn_mysql = None
    cursor = None

    try:
        # --- اتصال ---
        print("اتصال به SQL Server...")
        conn_sql = get_sql_connection()
        print("اتصال به MySQL...")
        conn_mysql = get_mysql_connection()

        # --- خواندن TimeStamps موجود ---
        print("خواندن TimeStamps موجود...")
        existing_ts = set()
        with conn_mysql.cursor() as c:
            c.execute("SELECT TimeStamps FROM results_dsas_mhi_lube_oil_11 WHERE unitID = 11")
            for row in c.fetchall():
                existing_ts.add(row['TimeStamps'])
        print(f"{len(existing_ts)} رکورد موجود است.")

        # --- خواندن 10 رکورد آخر برای 8341 ---
        print("خواندن 10 رکورد آخر برای AssetID=8341...")
        query_8341 = """
        SELECT TOP 10 [Value], [RecordTime], [RecordDate], [DateTime], [TimeStamp]
        FROM [PEGAH].[PDA].[Periodic_Values]
        WHERE [UnitID] = 11 AND [AssetID] = 8341
        ORDER BY [DateTime] DESC
        """
        cursor = conn_sql.cursor()
        cursor.execute(query_8341)
        rows_8341 = cursor.fetchall()

        if not rows_8341:
            print("هیچ داده‌ای برای AssetID=8341 پیدا نشد.")
            return

        inserted_count = 0
        for idx, row in enumerate(rows_8341):
            ts = row.TimeStamp
            if ts in existing_ts:
                print(f"رکورد {idx+1}: TimeStamps={ts} تکراری → رد شد.")
                continue

            dt = row.DateTime
            print(f"\n--- درج رکورد {idx+1} | TimeStamps={ts} ---")

            # درج اولیه
            insert_sql = """
            INSERT INTO results_dsas_mhi_lube_oil_11 
            (unitID, AssetID_8341, RecordTime, RecordDate, DateTime, TimeStamps)
            VALUES (%s, %s, %s, %s, %s, %s)
            """
            try:
                with conn_mysql.cursor() as c:
                    c.execute(insert_sql, (11, row.Value, 
                                         dt.strftime('%H:%M:%S'), dt.strftime('%Y-%m-%d'), dt, ts))
                conn_mysql.commit()
                print("رکورد درج شد.")
                inserted_count += 1
            except Exception as e:
                print(f"خطا در درج: {e}")
                continue

            # آپدیت 6 فیلد دیگر
            for aid in [8342, 8343, 8344, 8346, 9286, 9287]:
                field = field_map[aid]
                near_sql = f"""
                SELECT TOP 1 [Value] FROM [PEGAH].[PDA].[Periodic_Values]
                WHERE [UnitID]=11 AND [AssetID]=? AND ABS([TimeStamp] - ?) < 1000
                ORDER BY ABS([TimeStamp] - ?)
                """
                cursor.execute(near_sql, (aid, ts, ts))
                near_row = cursor.fetchone()

                if near_row and near_row.Value is not None:
                    value = float(near_row.Value)
                    source = "نزدیک"
                else:
                    avg_sql = f"""
                    SELECT AVG(CAST([Value] AS FLOAT)) FROM (
                        SELECT TOP 10 [Value] FROM [PEGAH].[PDA].[Periodic_Values]
                        WHERE [UnitID]=11 AND [AssetID]=?
                        ORDER BY [DateTime] DESC
                    ) sub
                    """
                    cursor.execute(avg_sql, (aid,))
                    avg_row = cursor.fetchone()
                    value = round(float(avg_row[0]), 4) if avg_row and avg_row[0] else None
                    source = "میانگین"

                if value is not None:
                    update_sql = f"UPDATE results_dsas_mhi_lube_oil_11 SET {field} = %s WHERE TimeStamps = %s"
                    try:
                        with conn_mysql.cursor() as c:
                            c.execute(update_sql, (value, ts))
                        conn_mysql.commit()
                        print(f"   {field} = {value} ({source})")
                    except Exception as e:
                        print(f"   خطا در آپدیت {field}: {e}")
                else:
                    print(f"   {field}: داده‌ای پیدا نشد.")

        print(f"\nنتیجه این دور: {inserted_count} رکورد جدید درج و آپدیت شد.")

    except Exception as e:
        print(f"خطای کلی: {e}")
    finally:
        if cursor: cursor.close()
        if conn_sql: conn_sql.close()
        if conn_mysql: conn_mysql.close()

# ================== حلقه اصلی (هر 5 دقیقه) ==================
def scheduler():
    while RUNNING:
        run_task()
        print(f"\nمنتظر {INTERVAL_SECONDS//60} دقیقه بعدی...")
        # خوابیدن با چک کردن هر ثانیه (برای Ctrl+C)
        for _ in range(INTERVAL_SECONDS):
            if not RUNNING:
                break
            time.sleep(1)

# ================== توقف با Ctrl+C ==================
def signal_handler(sig, frame):
    global RUNNING
    print('\nدریافت سیگنال توقف (Ctrl+C). در حال خروج...')
    RUNNING = False

signal.signal(signal.SIGINT, signal_handler)

# ================== اجرا ==================
if __name__ == "__main__":
    print("اسکریپت هر 5 دقیقه یکبار اجرا میشود...")
    print("برای توقف، Ctrl+C بزنید.")
    scheduler()
    print("اسکریپت با موفقیت متوقف شد.")

اسکریپت هر 5 دقیقه یکبار اجرا میشود...
برای توقف، Ctrl+C بزنید.

اجرای وظیفه: 2025-11-12 11:00:47
اتصال به SQL Server...
اتصال به MySQL...
خواندن TimeStamps موجود...
0 رکورد موجود است.
خواندن 10 رکورد آخر برای AssetID=8341...

--- درج رکورد 1 | TimeStamps=1701564893 ---
رکورد درج شد.
   AssetID_8342 = 12.1 (نزدیک)
   AssetID_8343 = 69.0 (نزدیک)
   AssetID_8344 = -240.0 (نزدیک)
   AssetID_8346 = 6.9 (نزدیک)
   AssetID_9286 = 7.2 (نزدیک)
   AssetID_9287 = 1.28 (نزدیک)

--- درج رکورد 2 | TimeStamps=1701550623 ---
رکورد درج شد.
   AssetID_8342 = 12.1 (نزدیک)
   AssetID_8343 = 69.0 (نزدیک)
   AssetID_8344 = -240.0 (نزدیک)
   AssetID_8346 = 6.9 (نزدیک)
   AssetID_9286 = 7.2 (نزدیک)
   AssetID_9287 = 1.28 (نزدیک)

--- درج رکورد 3 | TimeStamps=1701537147 ---
رکورد درج شد.
